## **Trabajo Práctico Final: Bank Subscription — Parte 2: Preprocesamiento y manipulación de datos**
____
__Universidad Tecnológica Nacional, Buenos Aires__\
__Ingeniería Industrial__\
__Cátedra de Ciencia de Datos - ClusterAI 2026__
____
Grupo 3: Galdames, González, Molina, Resines.
- Nombre: **María**
- Apellido: **González**
- Legajo: **259571-0**
____

En este notebook se documentan **todas las manipulaciones** aplicadas a los datos antes de entrenar los modelos, justificadas con lo observado en el EDA:

1. Limpieza: renombre de columnas, tipos de datos y chequeo de duplicados.
2. Reconstrucción determinística de `pdays` y creación de la variable `contacto_previo`.
3. Separación **estratificada** en train (80%) y test (20%) **antes** de imputar o escalar (para evitar *data leakage*).
4. Definición del preprocesador: imputación (mediana / moda), **estandarización** (StandardScaler) y **one-hot encoding**, ajustado solo con train.
5. Verificación del resultado y guardado de los archivos que usa el notebook de machine learning.

Al ejecutar este notebook se generan `datos/bank_train.csv` y `datos/bank_test.csv`.

In [1]:
import pandas as pd                                          # dataframes
import numpy as np                                           # cálculo numérico
import matplotlib.pyplot as plt                              # gráficos
import seaborn as sns                                        # gráficos estadísticos
from sklearn.model_selection import train_test_split         # split train/test
from sklearn.pipeline import Pipeline                        # encadenar pasos
from sklearn.compose import ColumnTransformer                # distinto tratamiento por columna
from sklearn.impute import SimpleImputer                     # imputación de nulos
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # escalado y dummies

sns.set_context("talk", font_scale=0.7)
sns.set_style("ticks")
RANDOM_STATE = 42                                            # semilla para reproducibilidad

# 1. Carga y limpieza

In [2]:
bank = pd.read_csv('datos/bank_subscription.csv', sep=';')   # dataset original
print('Shape original:', bank.shape)

Shape original: (45211, 17)


In [3]:
# diccionario para renombrar las columnas a nombres cortos, en minúscula y sin espacios
renombrar = {
    'Age': 'age',
    'Job': 'job',
    'Marital Status': 'marital',
    'Education': 'education',
    'Credit': 'credit_default',
    'Balance (euros)': 'balance',
    'Housing Loan': 'housing_loan',
    'Personal Loan': 'personal_loan',
    'Contact': 'contact',
    'Last Contact Day': 'day',
    'Last Contact Month': 'month',
    'Last Contact Duration': 'duration',
    'Campaign': 'campaign',
    'Pdays': 'pdays',
    'Previous': 'previous',
    'Poutcome': 'poutcome',
    'Subscription': 'subscription',
}

bank = bank.rename(columns=renombrar)
bank.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'credit_default',
 'balance',
 'housing_loan',
 'personal_loan',
 'contact',
 'day',
 'month',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'subscription']

In [4]:
# chequeamos duplicados y valores fuera de rango
print('Duplicados:', bank.duplicated().sum())
print('Edad mín/máx:', bank['age'].min(), bank['age'].max())
print('Día mín/máx:', bank['day'].min(), bank['day'].max())
print('Duración negativa:', (bank['duration'] < 0).sum())
print('Campaign < 1:', (bank['campaign'] < 1).sum())

Duplicados: 0
Edad mín/máx: 18.0 95.0
Día mín/máx: 1 31
Duración negativa: 0
Campaign < 1: 0


In [5]:
# eliminamos espacios accidentales en las categóricas (por seguridad)
cat_cols = ['job', 'marital', 'education', 'credit_default', 'housing_loan',
            'personal_loan', 'contact', 'month', 'poutcome']
for c in cat_cols:
    bank[c] = bank[c].str.strip()

# nulos antes de las manipulaciones
nulos_antes = bank.isnull().sum()
nulos_antes[nulos_antes > 0]

age               4973
job               4973
marital           4973
education         4973
credit_default    4973
balance           4973
housing_loan      7686
personal_loan     7686
duration          7686
pdays             7686
dtype: int64

# 2. Reconstrucción de `pdays` y feature engineering
*Feature engineering* (clase 02): construir nuevas variables a partir de las existentes para que el modelo aprenda mejor.

En el EDA vimos que cuando `previous == 0` (el cliente nunca fue contactado antes), `pdays` vale **siempre -1**. Usamos esa regla para completar los nulos de `pdays` de esos clientes: no es una imputación estadística sino información que ya está en el dataset.

In [6]:
mask = bank['pdays'].isnull() & (bank['previous'] == 0)   # nulos de pdays en clientes sin contacto previo
print('Nulos de pdays reconstruidos como -1:', mask.sum())
bank.loc[mask, 'pdays'] = -1
print('Nulos de pdays restantes (se imputarán):', bank['pdays'].isnull().sum())

Nulos de pdays reconstruidos como -1: 6269
Nulos de pdays restantes (se imputarán): 1417


In [7]:
# nueva variable binaria: ¿el cliente fue contactado en una campaña anterior?
# está siempre disponible (previous no tiene nulos) y separa el -1 "artificial" de pdays
bank['contacto_previo'] = (bank['previous'] > 0).astype(int)
bank.groupby('contacto_previo')['subscription'].agg(['mean', 'count'])

,mean,count
contacto_previo,,
0,0.091573,36954
1,0.230713,8257


In [8]:
# la edad es entera: la pasamos a tipo Int (admite nulos)
bank['age'] = bank['age'].round().astype('Int64')
bank.dtypes

age                  Int64
job                    str
marital                str
education              str
credit_default         str
balance            float64
housing_loan           str
personal_loan          str
contact                str
day                  int64
month                  str
duration           float64
campaign             int64
pdays              float64
previous             int64
poutcome               str
subscription         int64
contacto_previo      int64
dtype: object

**Variables que no se modifican:**
- Los *outliers* se mantienen (son valores reales, ver EDA).
- La categoría `unknown` de `job`, `education`, `contact` y `poutcome` se conserva como una categoría más.
- `duration` se conserva en el dataset; en el notebook de ML se evalúa el efecto de excluirla.

# 3. Separación train / test (estratificada)
Se separa **antes** de imputar y escalar: las medianas, modas, medias y desvíos se calculan **solo con train** y luego se aplican a test. Así el set de test simula datos nuevos que el modelo nunca vio.

In [9]:
target = 'subscription'
X = bank.drop(columns=target)        # variables explicativas
y = bank[target]                     # variable objetivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)   # stratify mantiene la proporción de clases

print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Proporción de suscriptos en train: {:.2%}'.format(y_train.mean()))
print('Proporción de suscriptos en test:  {:.2%}'.format(y_test.mean()))

Train: (36168, 17) | Test: (9043, 17)
Proporción de suscriptos en train: 11.70%
Proporción de suscriptos en test:  11.70%


## Imputación condicional de `pdays`
Los nulos de `pdays` que quedan son de clientes **con** contacto previo (`previous > 0`), cuyo valor real es > 0. Imputarlos con la mediana global daría -1 ("nunca contactado"), lo cual sería contradictorio. Por eso se imputan con la **mediana de `pdays` de los clientes contactados previamente en train**.

In [10]:
mediana_pdays_contactados = X_train.loc[X_train['previous'] > 0, 'pdays'].median()   # calculada solo con train
print('Mediana de pdays (clientes con contacto previo, train):', mediana_pdays_contactados)
for df_ in (X_train, X_test):
    df_['pdays'] = df_['pdays'].fillna(mediana_pdays_contactados)
print('Nulos de pdays en train / test:', X_train['pdays'].isnull().sum(), '/', X_test['pdays'].isnull().sum())

Mediana de pdays (clientes con contacto previo, train): 194.0
Nulos de pdays en train / test: 0 / 0


# 4. Preprocesador: imputación + escalado + one-hot encoding
Siguiendo lo visto en las clases 02 (pre-procesamiento) y 07 (ML Workflow):

- **Variables categóricas → dummies (one-hot encoding).** Cada categoría se transforma en una nueva variable binaria (1 si el cliente pertenece a esa categoría, 0 si no). Se usa en lugar de asignar números (0, 1, 2...) porque las categorías no tienen un orden natural. `handle_unknown='ignore'` evita errores si en test aparece una categoría que no estaba en train.
- **Standardization / auto-scaling (Z-score).** $z = \frac{x - \mu}{\sigma}$: cada variable numérica queda con media 0 y desvío 1. Sin esto, variables de gran magnitud (como `balance`, en miles de euros) dominarían las distancias de KNN, los pesos de la regresión logística/red neuronal y la varianza que captura PCA. Se elige Z-score en vez de Min-Max porque Min-Max es muy sensible a los outliers (que decidimos conservar).
- **Imputación.** Mediana para numéricas (robusta a outliers) y moda para categóricas.
- **Sin data leakage.** Los transformers son *stateful*: aprenden algo en `.fit()` (medianas, modas, $\mu$, $\sigma$, categorías). Por eso se ajustan **solo con train** y luego se aplican con `.transform()` a test, conservando el objeto ajustado.
- **`ColumnTransformer` + `Pipeline`.** Aplican en paralelo una secuencia de pasos distinta a las columnas numéricas y a las categóricas, y concatenan el resultado.

In [11]:
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous', 'contacto_previo']

# numéricas: imputar con la mediana (robusta a outliers) y estandarizar (media 0, desvío 1)
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),
                     ('scaler', StandardScaler())])
# categóricas: imputar con la moda y transformar a variables dummies (one-hot)
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                     ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocesador = ColumnTransformer([('num', num_pipe, num_cols),
                                   ('cat', cat_pipe, cat_cols)])
preprocesador

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [12]:
# ajustamos (fit) SOLO con train y transformamos train y test
X_train_proc = preprocesador.fit_transform(X_train.astype({'age': float}))
X_test_proc = preprocesador.transform(X_test.astype({'age': float}))

nombres = preprocesador.get_feature_names_out()          # nombres de las columnas resultantes
X_train_proc = pd.DataFrame(X_train_proc, columns=nombres, index=X_train.index)
X_test_proc = pd.DataFrame(X_test_proc, columns=nombres, index=X_test.index)
print('Dimensión luego del preprocesamiento:', X_train_proc.shape[1], 'features')
X_train_proc.head()

Dimensión luego del preprocesamiento: 52 features


,num__age,num__balance,num__day,num__duration,num__campaign,num__pdays,num__previous,num__contacto_previo,cat__job_admin.,cat__job_blue-collar,...,cat__month_jun,cat__month_mar,cat__month_may,cat__month_nov,cat__month_oct,cat__month_sep,cat__poutcome_failure,cat__poutcome_other,cat__poutcome_success,cat__poutcome_unknown
24001,-0.466466,-0.138119,1.582124,-0.443988,-0.246104,-0.417613,-0.241509,-0.471755,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
43409,-1.657763,0.972472,-1.298384,2.782345,0.398202,1.523183,2.664584,2.119745,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
20669,0.327732,-0.347991,-0.458236,6.265269,0.398202,-0.417613,-0.241509,-0.471755,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
18810,0.724831,-0.430988,1.822166,-0.271524,2.653271,-0.417613,-0.241509,-0.471755,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
23130,-0.267917,-0.343569,1.222060,-0.793121,2.331118,-0.417613,-0.241509,-0.471755,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [13]:
# valores aprendidos en train para imputar
imp_num = preprocesador.named_transformers_['num'].named_steps['imputer']
imp_cat = preprocesador.named_transformers_['cat'].named_steps['imputer']
print('Medianas usadas:', dict(zip(num_cols, imp_num.statistics_)))
print('Modas usadas:   ', dict(zip(cat_cols, imp_cat.statistics_)))

Medianas usadas: {'age': np.float64(39.0), 'balance': np.float64(450.0), 'day': np.float64(16.0), 'duration': np.float64(181.0), 'campaign': np.float64(2.0), 'pdays': np.float64(-1.0), 'previous': np.float64(0.0), 'contacto_previo': np.float64(0.0)}
Modas usadas:    {'job': 'blue-collar', 'marital': 'married', 'education': 'secondary', 'credit_default': 'no', 'housing_loan': 'yes', 'personal_loan': 'no', 'contact': 'cellular', 'month': 'may', 'poutcome': 'unknown'}


# 5. Verificaciones

In [14]:
print('Nulos en train procesado:', int(X_train_proc.isnull().sum().sum()))
print('Nulos en test procesado: ', int(X_test_proc.isnull().sum().sum()))

# las numéricas en train quedan con media 0 y desvío 1; en test no exactamente (el scaler se ajustó con train)
cols_num = [f'num__{c}' for c in num_cols]
pd.DataFrame({'media_train': X_train_proc[cols_num].mean(), 'std_train': X_train_proc[cols_num].std(),
              'media_test': X_test_proc[cols_num].mean(), 'std_test': X_test_proc[cols_num].std()}).round(3)

Nulos en train procesado: 0
Nulos en test procesado:  0


,media_train,std_train,media_test,std_test
num__age,0.0,1.0,0.009,0.987
num__balance,0.0,1.0,-0.001,0.964
num__day,0.0,1.0,-0.007,0.994
num__duration,0.0,1.0,-0.005,0.978
num__campaign,0.0,1.0,-0.000,0.990
num__pdays,0.0,1.0,0.007,1.012
num__previous,-0.0,1.0,-0.003,0.757
num__contacto_previo,-0.0,1.0,0.008,1.006


# 6. Guardado

Se guardan los sets de train y test **limpios pero sin imputar/escalar**. El notebook de machine learning vuelve a aplicar este mismo preprocesador **dentro de un `Pipeline` de scikit-learn**, de modo que en la validación cruzada la imputación y el escalado se ajustan en cada *fold* solo con los datos de entrenamiento de ese fold (sin *leakage*). También se guarda la versión ya procesada como referencia.

In [15]:
train = X_train.assign(subscription=y_train)
test = X_test.assign(subscription=y_test)
train.to_csv('datos/bank_train.csv', index=False)
test.to_csv('datos/bank_test.csv', index=False)

print('Archivos guardados en la carpeta datos/')

Archivos guardados en la carpeta datos/


# Resumen de manipulaciones
| Paso | Decisión | Justificación |
|---|---|---|
| Renombre | Nombres cortos en minúscula | Legibilidad |
| `pdays` nulo con `previous = 0` | Se reemplaza por -1 | Regla determinística del dataset |
| `pdays` nulo con `previous > 0` | Mediana de `pdays` de los contactados (train) | Evita asignar -1 a clientes que sí fueron contactados |
| `contacto_previo` | Nueva variable binaria `previous > 0` | Separa el -1 de pdays y no tiene nulos |
| Split | 80/20 estratificado, `random_state = 42` | Mantener 11,7% de positivos en ambos sets |
| Nulos numéricos | Mediana (de train) | Nulos dispersos al azar; la mediana es robusta a outliers |
| Nulos categóricos | Moda (de train) | Nulos dispersos al azar |
| Escalado | StandardScaler (de train) | KNN, regresión logística, redes y PCA son sensibles a la escala |
| Categóricas | One-hot encoding | Los modelos requieren entradas numéricas; no hay orden natural |
| Outliers | Se conservan | Valores reales |